<div style='background:#1a1a2e;color:#eee;padding:18px;border-radius:8px;margin-bottom:12px'>
<h2 style='margin:0'>🍎 EDUCATOR VERSION — Chapter 12: Counting Principles</h2>
<p style='margin:6px 0 0'>This notebook contains answer keys, teaching notes, and discussion prompts. Students should use the standard <strong>Chapter_12.ipynb</strong>.</p>
</div>


# Chapter 12: Counting Principles in Structural Engineering
## How Many Ways Can a Structure Be Loaded?

**Learning Objectives:**
- Apply the multiplication rule for counting to structural load combinations
- Use combinations to count truss member arrangements and inspection schedules
- Explain why engineers must check *every* load combination — and what happens when one is missed


<center>
<img src='https://upload.wikimedia.org/wikipedia/commons/thumb/1/16/Hartford_Civic_Center_Coliseum.jpg/800px-Hartford_Civic_Center_Coliseum.jpg' width='650' />

<em>The Hartford Civic Center, Hartford, Connecticut. The roof of the arena collapsed in January 1978 under accumulated snow and dead load — eighteen hours after 5,000 fans left a basketball game. (Wikimedia Commons — CC BY-SA.)</em>
</center>

---


<div style='background:#fff3cd;padding:15px;border-left:5px solid #ffc107;margin:10px 0'>

### 🍎 Teaching Context

**Curriculum connections:**
- *Statistics class*: Multiplication counting principle; combinations vs. permutations; sample space enumeration
- *Math/precalculus class*: Combinations C(n,k); factorial notation; systematic enumeration
- *Physics class*: Forces as vectors with direction — why load combinations matter physically

**Prerequisites:** Students should be comfortable with basic multiplication and have seen factorial notation. No prior knowledge of structural loads is assumed.

**Estimated time:** 45–50 minutes. The load combination widget (Widget 1) works well as a whole-class demo. Widget 2 (truss member inspection schedules) is best as independent or pair work.

**Pedagogical note:** The key insight is that combining loads is *not* just addition — loads can act simultaneously in ways that create worse conditions than any single load alone. The ASCE 7 load combinations encode decades of engineering judgment about which combinations are credibly simultaneous. The Hartford collapse is compelling precisely because it looks obvious in hindsight: snow + self-weight + vibration. Ask students: which combination would *you* have checked if you'd been the engineer?
</div>


In [ ]:
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','ipywidgets','--quiet'])
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display
from itertools import combinations as combs
import math
%matplotlib inline
print('Libraries loaded.')


## 12.1  The Multiplication Rule in Structural Design

When an engineer designs a structure, they must ask: *what combinations of loads could act on this structure at the same time?*

Hibbeler §1.3 identifies the primary load types a structure must resist. The ASCE/SEI 7-10 standard — referenced throughout Chapter 1 — specifies that structural members must be designed to resist the most demanding of **seven basic load combinations**. Each combination multiplies the individual loads by *load factors* that account for uncertainty:

| Combination | Formula |
|-------------|--------|
| 1 | 1.4D |
| 2 | 1.2D + 1.6L + 0.5(Lr or S or R) |
| 3 | 1.2D + 1.6(Lr or S or R) + (L or 0.5W) |
| 4 | 1.2D + 1.0W + L + 0.5(Lr or S or R) |
| 5 | 0.9D + 1.0W |
| 6 | 1.2D + 1.0E + L + 0.2S |
| 7 | 0.9D + 1.0E |

Where **D** = dead, **L** = live, **Lr** = roof live, **S** = snow, **R** = rain, **W** = wind, **E** = earthquake.

**Why seven combinations?** Because each load type has a different probability of being at its maximum simultaneously with other loads. The multiplication rule for counting tells us the *maximum possible* number of combinations — and the code's job is to select which ones actually govern design.


## 🔬 Interactive Experiment 1: Building Your Own Load Combination

Use the checkboxes below to select which load types are present at your site (e.g., a building in a snowy, windy region near a fault line has all of them). The widget shows which ASCE 7-10 combinations apply and what the total factored load looks like.


In [ ]:
LOAD_FACTORS = {
    'Dead (D)':         {'D':1.2, 'combo':'Standard'},
    'Live (L)':         {'L':1.6, 'combo':'Occupancy'},
    'Roof Live (Lr)':   {'Lr':1.6,'combo':'Roof'},
    'Snow (S)':         {'S':1.6, 'combo':'Roof'},
    'Wind (W)':         {'W':1.0, 'combo':'Lateral'},
    'Earthquake (E)':   {'E':1.0, 'combo':'Seismic'},
    'Rain (R)':         {'R':1.6, 'combo':'Roof'},
}

# Representative nominal loads (kips) — loosely based on the two-story office building
# discussed in Hibbeler §1.2-1.3 context (dead load from floor/roof system,
# live load from ASCE 7-10 Table 4-1 office occupancy). Values are illustrative;
# Hibbeler does not tabulate a single canonical example with all seven load types.
NOMINAL = {'D':40, 'L':20, 'Lr':10, 'S':15, 'W':12, 'E':18, 'R':8}

COMBINATIONS = [
    ('1',  lambda s: 1.4*s.get('D',0)),
    ('2',  lambda s: 1.2*s.get('D',0)+1.6*s.get('L',0)+0.5*max(s.get('Lr',0),s.get('S',0),s.get('R',0))),
    ('3',  lambda s: 1.2*s.get('D',0)+1.6*max(s.get('Lr',0),s.get('S',0),s.get('R',0))+max(s.get('L',0),0.5*s.get('W',0))),
    ('4',  lambda s: 1.2*s.get('D',0)+1.0*s.get('W',0)+s.get('L',0)+0.5*max(s.get('Lr',0),s.get('S',0),s.get('R',0))),
    ('5',  lambda s: 0.9*s.get('D',0)+1.0*s.get('W',0)),
    ('6',  lambda s: 1.2*s.get('D',0)+1.0*s.get('E',0)+s.get('L',0)+0.2*s.get('S',0)),
    ('7',  lambda s: 0.9*s.get('D',0)+1.0*s.get('E',0)),
]

checks = {k: widgets.Checkbox(value=(k in ['Dead (D)','Live (L)']), description=k,
    layout=widgets.Layout(width='200px')) for k in LOAD_FACTORS}

def _combo_plot(**kwargs):
    selected = {k.split(' ')[0].rstrip('(').lstrip(): NOMINAL[k.split('(')[1].rstrip(')')] 
                for k,v in kwargs.items() if v}
    combos_vals = [(name, fn(selected)) for name, fn in COMBINATIONS]
    names = [f'Combo {c[0]}' for c in combos_vals]
    vals  = [c[1] for c in combos_vals]
    governing_idx = int(np.argmax(vals))

    fig, ax = plt.subplots(figsize=(10, 5))
    colors = ['firebrick' if i == governing_idx else 'steelblue' for i in range(len(vals))]
    bars = ax.bar(names, vals, color=colors, edgecolor='white', width=0.6)
    ax.set_ylabel('Factored Load (kips)', fontsize=12)
    ax.set_title('ASCE 7-10 Load Combinations — Which Governs?', fontsize=13)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1, f'{val:.0f}',
            ha='center', fontsize=10)
    ax.annotate(f'Governing: Combo {combos_vals[governing_idx][0]}  ({vals[governing_idx]:.0f} kips)',
        xy=(0.5, 0.93), xycoords='axes fraction', ha='center', fontsize=11,
        bbox=dict(boxstyle='round', facecolor='mistyrose', alpha=0.9))
    plt.tight_layout()
    plt.show()

box_grid = widgets.GridBox(list(checks.values()), layout=widgets.Layout(grid_template_columns='repeat(3,210px)'))
out1 = widgets.interactive_output(_combo_plot, {k: v for k,v in checks.items()})
display(widgets.VBox([box_grid, out1]))


---
## ⚠️  Real-World Case: The Hartford Civic Center Roof Collapse (1978)

On January 18, 1978 — just 18 hours after 5,000 fans left a University of Connecticut basketball game — the roof of the Hartford Civic Center collapsed. No one was killed. The structure had been open for less than three years.

**What went wrong:**

The roof was a two-way space frame: a three-dimensional truss system spanning 300 ft × 360 ft with no interior columns. The design engineers used a computer analysis program — cutting-edge for 1973 — to check the structure.

The investigation found three compounding failures, all rooted in counting:

1. **The dead load was underestimated** — the computer model used a self-weight that was 20% lower than the structure as actually built. Electrical equipment and mechanical systems added to the roof were never added to the load model.

2. **The critical load combination was never checked** — the governing case of dead load + snow load + ponded water (from melting and refreezing) produced a load far exceeding the design capacity. This combination existed in the ASCE standards but was not checked in the computer run.

3. **The slender compression members buckled** — the top chord members of the space frame were slender enough that they were governed by buckling (Euler's formula, which Hibbeler covers in later chapters), not direct compression strength. The analysis used the wrong failure mode.

The roof had been deflecting visibly for months — workers had placed buckets to catch leaks from the sagging panels. No one connected the dots.

> *The multiplication rule of counting works in both directions: the number of combinations you must check is large. Missing even one can be fatal.*


<div style='background:#fff3cd;padding:15px;border-left:5px solid #ffc107;margin:10px 0'>

### 🍎 Teaching Note — Hartford Civic Center Case Study

The Hartford Civic Center roof collapse (1978) is a textbook case of *missed combination*. The space truss roof was designed in the early 1970s, before the widespread adoption of computer-aided structural analysis. The design engineers checked gravity loads and snow loads separately — but the combination of:
1. Higher-than-expected self-weight (original estimates were low)
2. Accumulated wet snow (the storm the night of collapse)
3. Vibration from HVAC equipment and crowd loads

...exceeded the capacity of the slender compression members in the top chord.

**Key point for students:** No individual load caused the collapse. The failure was combinatorial — it required several loads to act together. This is exactly the problem the ASCE 7 load combination table was designed to force engineers to check systematically.

**Fortunate timing:** The roof collapsed at 4:19 AM, hours after a UConn basketball game that had drawn 5,000 spectators. Had it collapsed during the game, casualties would have been severe.

**Physics connection:** The failure was buckling of slender compression members (Euler buckling: P_cr = π²EI/L²). The members were undersized for the actual combined load. Ask students: if a column's buckling load is 80 kips, and the combined factored load reaches 95 kips, what happens?
</div>


## 🔬 Interactive Experiment 2: How Many Ways Can You Inspect a Truss?

A simple Pratt truss bridge has many individual members. An inspector has time to thoroughly check only a subset of them on a given visit. The combination formula C(n, k) = n! / (k! × (n-k)!) tells us how many different inspection subsets are possible.

Use the sliders to explore how the number of possible inspection schedules grows — and why a *systematic* sampling strategy matters more than a random one when resources are limited.


In [ ]:
def _truss_combos(n_members, k_inspected):
    if k_inspected > n_members:
        k_inspected = n_members
    n_combos = math.comb(n_members, k_inspected)
    pct = k_inspected / n_members * 100

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Left: bar of inspected vs not
    ax1.bar(['Inspected', 'Not Inspected'], [k_inspected, n_members - k_inspected],
        color=['steelblue', 'lightgray'], edgecolor='white', width=0.5)
    ax1.set_ylabel('Number of Members', fontsize=12)
    ax1.set_title(f'Truss Inspection Coverage  ({pct:.0f}%)', fontsize=13)
    ax1.annotate(f'{k_inspected} of {n_members} members\ninspected this visit',
        xy=(0.5, 0.85), xycoords='axes fraction', ha='center', fontsize=11,
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))

    # Right: how combos grow with k
    ks = list(range(1, n_members+1))
    cs = [math.comb(n_members, k) for k in ks]
    ax2.bar(ks, cs, color=['firebrick' if k == k_inspected else 'steelblue' for k in ks],
        edgecolor='white')
    ax2.set_xlabel('Members Inspected (k)', fontsize=12)
    ax2.set_ylabel('Possible Inspection Schedules  C(n,k)', fontsize=12)
    ax2.set_title(f'C({n_members}, k) — Number of Ways to Choose k Members', fontsize=13)
    ax2.annotate(f'C({n_members},{k_inspected}) = {n_combos:,}',
        xy=(0.55, 0.85), xycoords='axes fraction', fontsize=11,
        bbox=dict(boxstyle='round', facecolor='mistyrose', alpha=0.9))
    plt.tight_layout()
    plt.show()

w_n = widgets.IntSlider(value=12, min=5, max=30, step=1,
    description='Truss members (n):', style={'description_width':'initial'},
    layout=widgets.Layout(width='400px'))
w_k = widgets.IntSlider(value=4, min=1, max=20, step=1,
    description='Members inspected (k):', style={'description_width':'initial'},
    layout=widgets.Layout(width='400px'))
out2 = widgets.interactive_output(_truss_combos, {'n_members': w_n, 'k_inspected': w_k})
display(widgets.VBox([w_n, w_k, out2]))


---
## 📋  Chapter 12 Review

| Concept | Structural Application |
|---------|----------------------|
| **Multiplication rule** | Number of ways k load types can combine = product of individual options |
| **Combinations C(n,k)** | Ways to select k members for inspection from n total; ways to select k load types from n available |
| **Permutations** | Order-sensitive counting — e.g., the sequence in which loads are applied during a proof load test |
| **Factorial n!** | Number of ways to order n inspection events |

**The Big Idea:** Counting tells engineers how large a problem space they are searching. The ASCE 7-10 load combinations are not arbitrary — they represent the code committee's answer to the combinatorics question: *which subsets of loads can plausibly occur simultaneously, and which combinations produce the worst demands?* When engineers skip combinations, they are not saving time. They are leaving part of the problem unchecked.


<div style='background:#d1ecf1;padding:15px;border-left:5px solid #17a2b8;margin:10px 0'>

### 💬 Discussion Prompts

**1. Combinations in daily life (warm-up):** A restaurant has 4 appetizers, 6 entrées, and 3 desserts. How many distinct meals can you order (one of each)? Now: what if you only want 2 of the 3 courses — how many options? *(The multiplication principle and combinations apply directly. This bridges the abstract math to something familiar before the structural context.)*

**2. Why factors differ (pair discussion):** Why does dead load get a factor of 1.2 but live load gets 1.6? What does it mean that we're *less* certain about live loads than dead loads? *(Dead load = weight of the structure itself, which is calculated from drawings and material densities — well-controlled. Live load = people, furniture, stored goods — highly variable and unpredictable.)*

**3. Missed combinations (small group):** The Hartford engineers didn't use the systematic combination checking that is now required. What organizational or cultural factors might lead a team to skip some combinations? *(Guide toward: time pressure, overconfidence, missing software, lack of peer review, assumption that 'it has always worked this way.')*

**4. Extension (homework):** Research the term 'system safety' and find one example of a multi-factor failure — in aviation, medicine, or nuclear power — where no single cause was sufficient to produce the accident. Write one paragraph explaining which factors combined to cause the failure.
</div>
